## 1. Importação e localização do projeto


In [ ]:
# Path organiza os caminhos de forma portável entre as máquinas da equipe.
from pathlib import Path
# sys.path permite localizar src ao abrir o notebook dentro da pasta notebooks.
import sys
# pandas formata as tabelas; matplotlib mostra e fecha as figuras.
import pandas as pd
import matplotlib.pyplot as plt
# display apresenta DataFrames e figuras no Jupyter ou no VS Code.
from IPython.display import display

# Funciona ao iniciar Jupyter na raiz SAC ou na pasta notebook.
# Parte do diretório de execução e identifica a pasta raiz que contém src.
RAIZ = Path.cwd().resolve()
# Se o kernel iniciou em notebooks, procura a raiz no diretório pai.
if not (RAIZ / "src").is_dir():
    # Parte do diretório de execução e identifica a pasta raiz que contém src.
    RAIZ = RAIZ.parent
# Se a raiz ainda não foi encontrada, interrompe e orienta a abrir a pasta correta.
if not (RAIZ / "src").is_dir():
    raise RuntimeError("Inicie o Jupyter na pasta SAC ou SAC/notebooks.")
# Adiciona a raiz uma única vez ao caminho de importação deste kernel.
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

# Importa as funções do projeto; as regras permanecem nos arquivos Python.
from src.entrada import carregar_configuracao
from src.services.fluxo_service import executar_analise
from src.services.relatorio_service import (AVISO, gerar_tabelas, gerar_graficos,
                                             exportar_pdf, salvar_graficos)
from src.repositories.relatorio_repository import novo_caminho_relatorio
# Carrega os destinos centralizados e o caminho do cadastro salvo.
from config.configuracoes import ARQUIVO_VAGA, PASTA_GRAFICOS
# Reforça que a classificação é apenas apoio à decisão do RH.
print(AVISO)


## 2. Carregar o cadastro feito no terminal


In [ ]:
# Antes desta célula, execute na raiz SAC, em um terminal:
# python -m src.entrada
# Usa o JSON local criado com python -m src.entrada no terminal.
configuracao = ARQUIVO_VAGA
# Interrompe esta etapa com uma orientação caso o cadastro ainda não exista.
if not configuracao.is_file():
    raise FileNotFoundError("Faça o cadastro no terminal: python -m src.entrada")
# Reconstrói a vaga validada e recupera o caminho dos PDFs escolhido pelo usuário.
vaga, pasta = carregar_configuracao(configuracao)
print("Vaga:", vaga.nome)
print("Pasta dos currículos:", pasta)
# Exibe os critérios carregados para conferência antes de analisar os currículos.
for requisito in vaga.requisitos:
    # Traduz o booleano para uma descrição compreensível ao recrutador.
    tipo = "Obrigatório" if requisito.obrigatorio else "Desejável"
    print(f"{requisito.nome} | {tipo} | {requisito.peso}% | {', '.join(requisito.palavras_chave)}")


## 3. Processar os currículos


In [ ]:
# Coordena localização, extração, análise e ranking por meio do serviço de aplicação.
lote = executar_analise(vaga, pasta)
# Conta somente currículos com leitura e análise concluídas.
print("Processados com sucesso:", lote.processados)
# Mostra quantos arquivos falharam sem interromper o lote inteiro.
print("Erros:", len(lote.erros))
# Conta ausências de obrigatórios; esses candidatos permanecem fora do ranking.
print("Eliminados da classificação:", sum(bool(r.obrigatorios_ausentes) for r in lote.resultados))
# Conta apenas resultados com posição atribuída pelo serviço de ranking.
print("Presentes no ranking:", sum(r.posicao is not None for r in lote.resultados))


## 4. Tabelas de resultados


In [ ]:
# Obtém ranking, detalhes, erros e frequências a partir da mesma análise.
tabelas = gerar_tabelas(lote)
# Apresenta cada visão dos resultados na sequência da demonstração.
for nome in ("ranking", "detalhada", "erros", "frequencias"):
    print(nome.upper())
    tabela = tabelas[nome]
    # Trata lotes ou tabelas sem registros com uma mensagem clara.
    if tabela.empty:
        print("Sem registros.")
    else:
        # Copia a tabela para formatar a apresentação sem alterar os valores originais.
        exibicao = tabela.copy()
        # Formata valores numéricos como porcentagem e trata os casos não aplicáveis.
        if "Aderência" in exibicao.columns:
            exibicao["Aderência"] = exibicao["Aderência"].map(
                lambda valor: "Não se aplica" if pd.isna(valor) else f"{valor:.2f}%")
        # Amplia a exibição apenas neste bloco para permitir a leitura dos requisitos.
        with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
            # Mostra a tabela e sinaliza informações que não se aplicam a eliminados ou falhas.
            display(exibicao.fillna("Não se aplica"))


## 5. Três gráficos


In [ ]:
# Solicita os três gráficos definidos no serviço de relatório.
figuras = gerar_graficos(lote)
# Apresenta cada figura no notebook e libera o recurso depois de exibi-la.
for figura in figuras:
    display(figura)
    plt.close(figura)


## 6. Exportação automática do PDF e dos gráficos


In [ ]:
# Exportação automática ao concluir o fluxo, conforme o guia técnico.
# As regras de análise não são executadas novamente nesta célula.
# Prepara um caminho de PDF com nome da vaga e identificador de data/hora.
destino = novo_caminho_relatorio(vaga.nome)
# Agrupa as gravações para informar falhas de acesso ou de escrita ao usuário.
try:
    # Exporta automaticamente critérios, resultados e erros da análise em memória.
    caminho_pdf = exportar_pdf(lote, destino)
    print("Relatório salvo em:", caminho_pdf)
    # Salva cópias PNG dos três gráficos na pasta de saída centralizada.
    caminhos_graficos = salvar_graficos(lote, PASTA_GRAFICOS)
    # Informa onde cada imagem foi gravada no computador.
    for caminho in caminhos_graficos:
        print("Gráfico salvo em:", caminho)
# Exibe o problema de gravação e mantém a exceção visível para diagnóstico.
except OSError as erro:
    print("Falha ao salvar a saída:", erro)
    raise
